# Chapter 12: Serial Correlation and Heteroscedasticity in Time Series

Topics: testing for serial correlation, FGLS, HAC standard errors, ARCH/GARCH.

## Step 1 – Setup: Working Directory & Output Folder

Run this cell first to fix the working directory and create the `output/` folder.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import wooldridge as woo
import matplotlib.pyplot as plt
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import acorr_breusch_godfrey, het_arch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import os

output_dir = "output"
os.makedirs(output_dir, exist_ok=True)
df = woo.dataWoo('phillips')
model = smf.ols('inf ~ unem', data=df).fit()

## 12.1 Testing for Serial Correlation

## Step 2 – What is Serial Correlation?

**Serial (autocorrelation):** $\text{Cov}(u_t, u_s) \neq 0$ for $t \neq s$.

**Consequences:** OLS is still unbiased but **standard errors are wrong** → inference is invalid.

**AR(1) error process:**
$$u_t = \rho u_{t-1} + e_t, \quad |\rho| < 1, \quad e_t \sim i.i.d.(0,\sigma_e^2)$$

**Durbin-Watson statistic:**
$$DW = \frac{\sum_{t=2}^n (\hat{u}_t - \hat{u}_{t-1})^2}{\sum_{t=1}^n \hat{u}_t^2} \approx 2(1-\hat{\rho})$$

Interpretation: DW ≈ 2 → no serial correlation; DW < 2 → positive AR(1).

In [ ]:
# Durbin-Watson
dw = durbin_watson(model.resid)
print(f"Durbin-Watson: {dw:.4f}")
print("DW ≈ 2: no autocorrelation | DW < 2: positive | DW > 2: negative")

# Breusch-Godfrey test
bg_stat, bg_pval, _, _ = acorr_breusch_godfrey(model, nlags=2)
print(f"\nBreusch-Godfrey (2 lags): stat={bg_stat:.4f}, p-value={bg_pval:.4f}")
print("→ Reject H0 of no serial correlation" if bg_pval < 0.05 else "→ Fail to reject H0")

## ACF and PACF of Residuals

## Step 3 – Breusch-Godfrey Test

The **Breusch-Godfrey test** allows for higher-order AR(p) errors and is valid when lagged $y$ appears as a regressor (unlike DW).

**Procedure:**
1. Run OLS → obtain $\hat{u}_t$
2. Regress $\hat{u}_t$ on $\mathbf{x}_t, \hat{u}_{t-1}, \ldots, \hat{u}_{t-p}$
3. $LM = n \cdot R^2 \sim \chi^2_p$ under H₀: no serial correlation

**ACF/PACF plots:** visual tools to identify the order of autocorrelation.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(model.resid,  ax=axes[0], lags=20, title='ACF of Residuals')
plot_pacf(model.resid, ax=axes[1], lags=20, title='PACF of Residuals')
plt.tight_layout()
plt.savefig(f"{output_dir}/fig1_acf_pacf.png", dpi=150)
plt.savefig(os.path.join(output_dir, 'fig_Chapter_12_Serial_Correlation_01.png'), dpi=150, bbox_inches='tight')
plt.show()

## 12.2 FGLS with AR(1) Errors

## Step 4 – FGLS and HAC Standard Errors

**FGLS (Cochrane-Orcutt):** transforms data to remove AR(1) serial correlation.

Quasi-differenced model:
$$\tilde{y}_t = y_t - \hat{\rho}y_{t-1}, \quad \tilde{x}_t = x_t - \hat{\rho}x_{t-1}$$

Then OLS on transformed data is efficient.

**Newey-West HAC standard errors** — robust to both serial correlation and heteroscedasticity:
$$\widehat{\text{Var}}_{NW}(\hat{\boldsymbol{\beta}}) = (\mathbf{X}'\mathbf{X})^{-1}\hat{\mathbf{S}}_{NW}(\mathbf{X}'\mathbf{X})^{-1}$$

In statsmodels: `results = model.fit(cov_type='HAC', cov_kwds={'maxlags': 4})`

In [ ]:
from statsmodels.regression.linear_model import GLSAR
import statsmodels.api as sm

y = df['inf'].values
X = sm.add_constant(df['unem'].values)

gls = GLSAR(y, X, rho=1)
gls_result = gls.iterative_fit(maxiter=10)
print("FGLS (Cochrane-Orcutt) Results:")
print(gls_result.summary())

## 12.3 HAC Standard Errors

In [ ]:
m_hac = smf.ols('inf ~ unem', data=df).fit(cov_type='HAC', cov_kwds={'maxlags': 2})
print("OLS with Newey-West HAC Standard Errors:")
print(m_hac.summary())

## 12.4 ARCH Test

In [ ]:
arch_stat, arch_pval, _, _ = het_arch(model.resid, nlags=4)
print(f"ARCH test (4 lags): stat={arch_stat:.4f}, p-value={arch_pval:.4f}")
print("→ Evidence of ARCH effects" if arch_pval < 0.05 else "→ No ARCH effects")